# Smoke-эксперимент: вероятность кредитного дефолта

## tl;dr

На 360 синтетических строках validation выбрал cost-порог 0.24. На независимой smoke-test части Brier score равен 0.1488, PR-AUC — 0.4668, precision — 0.4400, recall — 0.6875. Это техническая проверка pipeline, а не оценка кредитного риска или fairness реальной группы.

## Context & Methods

Notebook вызывает production-функции `train`, `evaluate` и `predict`. Внутри train используется split 60/20/20, калибровка только на train, выбор порога на validation и отчёт на test. `SEX` исключён из модели и используется для диагностических групповых срезов.

### Key Assumptions

- cost(FN)=5 и cost(FP)=1 — параметр smoke-сценария, не бизнес-оценка;
- синтетические категории и зависимости не описывают реальных клиентов;
- групповые разрывы без интервалов и нормативного критерия не доказывают fairness;
- вызов `evaluate` на полном smoke-файле ниже проверяет API, а итоговые числа берутся из held-out test внутри train.

In [1]:
import json
from pathlib import Path

from credit_default_risk_model.data import load_data, make_smoke_data
from credit_default_risk_model.evaluate import evaluate
from credit_default_risk_model.predict import predict
from credit_default_risk_model.train import train

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RUN_DIR = PROJECT_ROOT / "artifacts" / "notebook_smoke"
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = RUN_DIR / "credit.csv"
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/dviakhrameev/Documents/helen/credit-default-risk-model


## Data

Генератор с seed 42 создаёт UCI-подобную схему без реальных персональных данных. CSV повторно проходит публичный загрузчик и schema validation.

In [2]:
smoke = make_smoke_data(rows=360, seed=42)
smoke.to_csv(DATA_PATH, index=False)
validated = load_data(DATA_PATH)
data_summary = {
    "rows": len(validated),
    "default_rate": round(float(validated["default"].mean()), 4),
    "sex_groups": sorted(validated["SEX"].astype(int).unique().tolist()),
    "missing_feature_values": int(validated.isna().sum().sum()),
}
print(json.dumps(data_summary, ensure_ascii=False, indent=2))
validated.head(3)

{
  "rows": 360,
  "default_rate": 0.2278,
  "sex_groups": [
    1,
    2
  ],
  "missing_feature_values": 0
}


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,1,20000,2,3,2,60,-1,-1,0,-1,...,8059.25,8162.69,8642.29,2281.88,2045.39,1799.05,2081.59,1956.43,2153.48,0
1,2,200000,2,1,1,32,0,-1,0,0,...,75107.70,77809.34,66393.00,11654.67,14507.00,14594.46,7867.27,12654.91,10204.38,0
2,3,200000,1,1,1,40,-1,0,2,0,...,56526.95,57451.06,49812.25,10256.75,11386.56,7037.12,12567.19,13851.39,9358.63,0


## Results

Train сохраняет модель, held-out test predictions и JSON-метрики. Отдельные evaluate/predict вызовы проверяют возможность повторного использования артефакта; full-smoke evaluate не интерпретируется как независимая оценка.

In [3]:
train_metrics = train(DATA_PATH, RUN_DIR, false_negative_cost=5.0, false_positive_cost=1.0, seed=42)
api_evaluation = evaluate(DATA_PATH, RUN_DIR / "model.joblib")
scores = predict(DATA_PATH, RUN_DIR / "model.joblib")
result_summary = {
    "split": train_metrics["split"],
    "validation_threshold": round(train_metrics["validation_selected_threshold"], 4),
    "validation_mean_cost": round(train_metrics["validation_mean_cost"], 4),
    "held_out_test": {
        "brier_score": round(train_metrics["test"]["brier_score"], 4),
        "average_precision": round(train_metrics["test"]["average_precision"], 4),
        "precision": round(train_metrics["test"]["precision"], 4),
        "recall": round(train_metrics["test"]["recall"], 4),
    },
    "fairness_gaps_by_sex": train_metrics["fairness_by_sex"]["gaps"],
    "full_smoke_api_check_brier": round(api_evaluation["brier_score"], 4),
}
print(json.dumps(result_summary, ensure_ascii=False, indent=2))

{
  "split": {
    "train": 216,
    "validation": 72,
    "test": 72
  },
  "validation_threshold": 0.2594,
  "validation_mean_cost": 0.4583,
  "held_out_test": {
    "brier_score": 0.1488,
    "average_precision": 0.4668,
    "precision": 0.4762,
    "recall": 0.625
  },
  "fairness_gaps_by_sex": {
    "positive_rate_max_gap": 0.08901363271852447,
    "tpr_max_gap": 0.15873015873015872,
    "fpr_max_gap": 0.08333333333333334
  },
  "full_smoke_api_check_brier": 0.1445
}


In [4]:
assert sum(train_metrics["split"].values()) == len(validated)
assert scores["default_probability"].between(0, 1).all()
assert 0 < train_metrics["validation_selected_threshold"] < 1
assert (RUN_DIR / "test_predictions.csv").exists()
print("Leakage-boundary and artifact checks: OK")

Leakage-boundary and artifact checks: OK


## Takeaways

- Cost-порог 0.24 выбран только на validation; test не участвовал в выборе.
- Held-out smoke Brier 0.1488 и PR-AUC 0.4668 подтверждают выполнение расчётов, но не качество на UCI.
- Наблюдаемый smoke-разрыв TPR по `SEX` около 0.302 нельзя трактовать как доказательство дискриминации или её отсутствия; нужен реальный контекст, интервалы и governance.